# Multi-Head V_φ — OpenWebText Scale-Up (Fock-PARFLM v2.1)

## Motivation

Phase 5 (Gaussian / SARF wells) cured the SQ3 training instability but left a
large PPL gap versus a parameter-matched GPT-2 on OpenWebText (~4.5–5.3×; see
`companion_notes/Fock-PARFLM_vs_GPT-2_on_OpenWebText_Next_Steps.md`). The
diagnosis there is **pure underfitting** driven by **V_φ starvation**: a single
radial pair potential over a tiny interaction subspace
(`v_phi_d_type=16`, `v_phi_d_angle=8`) cannot express the directional,
content-dependent forces that attention provides.

This notebook tests **Bottleneck-1 cure A: multi-head V_φ**.

## What changes vs Phase 5

The single pair potential is replaced by a sum of `V_PHI_N_HEADS` independent
scalar pair sub-potentials:

> V_φ(h_i, h_j) = Σ_{m=1}^{H} V^(m)(h_i, h_j)

Each head owns its own type/angle projection, so H heads give the model an
effective H·d_type-dim type space and H distinct radial force directions.

**Conservativity is preserved.** A sum of scalar potentials is a scalar
potential, so F_i = −∇_{h_i} Σ_m V^(m) is still the gradient of a scalar (the
per-pair Jacobian stays symmetric — the hard constraint of the framework
holds). Cost: O(H) parameters; the sparse routing stays O(T·top_k) because all
heads share the single Gumbel-softmax top-k selection.

## Knobs (Cell 0)

| Knob | Meaning |
|------|---------|
| `V_THETA_VARIANT` | `gaussian` / `sarf` / `sq3` (one-body well family) |
| `K_MIX` | number of Gaussian / quadratic wells |
| `SARF_N_ANCHORS` | frozen PMI anchors (`sarf` only) |
| `XI_OVERRIDE` | `None` (4ch) or `5` (adds α=0.99 long-horizon channel) |
| `TOP_K` | sparse pairs per query (PARF routing) |
| `V_PHI_N_HEADS` | **new** — multi-head V_φ (`1` = Phase-5 baseline) |
| `USE_OUTPUT_BIAS` | **new** — learned logit bias, init to log-unigram-freq (D0.4 long-tail fix) |
| `TIE_EMBEDDINGS` | **new** — `True` reuses E^T read-out; `False` = dedicated W_out head |

## D0 diagnostic addendum (Xi=5 step-78k checkpoint)

The four D0 probes refined the diagnosis beyond "V_φ starvation alone":

- **D0.2 effective rank = 229.6 / d=384** — the hidden states are *not* collapsed
  (PR ≫ K_MIX), so the V_θ attractor count is **not** the binding constraint.
- **D0.3 per-step ‖Δh‖ is U-shaped** (late/early = 0.47) — depth is **used**, not
  wasted; late Verlet layers do real work.
- **D0.4 loss by frequency** — per-token CE on rare targets **exceeds ln(V)**
  (worse than uniform). With a tied head and no output bias the model collapses
  its output distribution onto the frequent-token prior. This is the **largest
  single mis-calibration** and is an output/embedding-head problem, not a
  context-mixing one.

→ This notebook therefore pairs the multi-head V_φ cure (the Q4-frequent-token
gap from D0.1) with the cheap **output-head fix** (`USE_OUTPUT_BIAS`,
`TIE_EMBEDDINGS`) that directly targets the D0.4 long tail. Both read-out knobs
sit outside the conservative force field, so conservativity is preserved.

Each distinct (`V_THETA_VARIANT`, `K_MIX`, `XI_OVERRIDE`, `TOP_K`,
`V_PHI_N_HEADS`) combination writes to its own checkpoint directory, so runs
never collide.

## Prerequisites

- OpenWebText tokenized and cached on Google Drive (reused from Phase 4/5)
- H100 80 GB GPU


In [ ]:
# ── Cell 0: Configuration ─────────────────────────────────────────
# V_THETA_VARIANT selects the potential architecture:
#   'gaussian'  — MixtureGaussianVTheta (bounded, K=K_MIX wells)
#   'sarf'      — SARFGaussianVTheta (bounded, frozen PMI anchors)
#   'sq3'       — MixtureQuadraticVTheta (unbounded, K=K_MIX wells)
V_THETA_VARIANT = 'gaussian'    # 'gaussian', 'sarf', or 'sq3'
K_MIX           = 8             # number of wells (8 = G1/S1, 16 = G2/S2)
SARF_N_ANCHORS  = 64            # for 'sarf' variant only
W_SCALE         = 1.0
BG_QUAD_EPS     = 0.0           # set > 0 if G5 wins TinyStories
SQ3_TAU         = 1.0           # temperature for SQ3 logsumexp mixture
SQ3_CURV_MAX    = 2.0           # max curvature per dim (None = unconstrained)

# ── Xi channel override ──────────────────────────────────────────────
# Set XI_OVERRIDE to change the number of Xi channels and their decays.
# None = use defaults (4 channels, [0.25, 0.50, 0.75, 0.95]).
# Set to 5 to add a long-horizon α=0.99 (~100 tok) channel for the
# bottleneck diagnostic experiment.
XI_OVERRIDE     = 5             # 5 = match the diagnosed Xi=5 run (adds α=0.99); None = 4ch

# ── PARF V_phi knobs ─────────────────────────────────────────
# TOP_K: number of past tokens each query routes to via Gumbel-softmax sparse
#   selection (pair-interaction cost is O(T * TOP_K)).  Was hardcoded to 8.
# V_PHI_N_HEADS: multi-head V_phi.  V_phi = sum_m V^(m): a sum of H independent
#   CONSERVATIVE scalar pair sub-potentials, each with its own d_type/d_angle
#   subspace.  Lifts the V_phi expressivity ceiling (Bottleneck 1) at O(H) param
#   cost and unchanged O(T * TOP_K) routing cost (heads share top-k selection).
#   1 = single-head baseline (identical to Phase 5).
TOP_K           = 8             # sparse pairs per query
V_PHI_N_HEADS   = 4             # 1 = baseline; >1 enables multi-head V_phi

# ── Output read-out head (D0.4 long-tail fix) ────────────────────────
# D0.4 on the Xi=5 step-78k checkpoint showed per-token CE on rare targets
# EXCEEDS ln(V) (worse than uniform): with a tied head and NO output bias the
# model must encode the entire unigram log-prior as a *direction* in h_L space,
# which it cannot do for the whole vocab at once, so rare-target positions
# default to the frequent-token direction.  Both knobs are pure post-dynamics
# read-out projections OUTSIDE the conservative force field (V_theta / V_phi),
# so neither affects conservativity.
#   USE_OUTPUT_BIAS: add a learned logit bias b_v, initialised to the log
#       unigram frequency (Cell 4) so the hidden dynamics stop spending
#       capacity on the frequency prior.  Cheap (~V params).
#   TIE_EMBEDDINGS:  True reuses E^T as the read-out (Phase-5 default; run this
#       bias-only variant FIRST since it is nearly free); set False to allocate
#       a dedicated W_out (V*d extra params) giving rare tokens an output
#       direction decoupled from their undertrained input embedding.
USE_OUTPUT_BIAS = True          # D0.4 fix: log-freq output bias (recommended)
TIE_EMBEDDINGS  = True          # False = untied dedicated W_out read-out head

# ── Variant tag: used for checkpoint dir & prefix so different
#    runs get separate checkpoint directories and don't conflict.
_variant_parts = []
if V_THETA_VARIANT == 'sq3':
    _variant_parts.append(f'sq3_k{K_MIX}')
elif V_THETA_VARIANT == 'gaussian' and K_MIX != 8:
    _variant_parts.append(f'k{K_MIX}')
if XI_OVERRIDE is not None:
    _variant_parts.append(f'xi{XI_OVERRIDE}')
if TOP_K != 8:
    _variant_parts.append(f'topk{TOP_K}')
if V_PHI_N_HEADS != 1:
    _variant_parts.append(f'mh{V_PHI_N_HEADS}')
if USE_OUTPUT_BIAS:
    _variant_parts.append('ob')
if not TIE_EMBEDDINGS:
    _variant_parts.append('untied')
_variant_tag = '_'.join(_variant_parts)

print(f'Phase 5 config: V_theta={V_THETA_VARIANT}')
if V_THETA_VARIANT == 'gaussian':
    print(f'  K_mix={K_MIX}, w_scale={W_SCALE}')
elif V_THETA_VARIANT == 'sq3':
    print(f'  K_mix={K_MIX}, tau={SQ3_TAU}, curvature_max={SQ3_CURV_MAX}')
else:
    print(f'  SARF N_S={SARF_N_ANCHORS}, w_scale={W_SCALE}')
if XI_OVERRIDE is not None:
    print(f'  XI_OVERRIDE={XI_OVERRIDE} channels')
print(f'  PARF top_k={TOP_K}, V_phi heads={V_PHI_N_HEADS}'
      + ('  (multi-head)' if V_PHI_N_HEADS > 1 else '  (single-head baseline)'))
print(f'  read-out: output_bias={USE_OUTPUT_BIAS}, '
      + ('tied E^T' if TIE_EMBEDDINGS else 'UNTIED W_out')
      + ('  (D0.4 long-tail fix)' if (USE_OUTPUT_BIAS or not TIE_EMBEDDINGS) else ''))
if _variant_tag:
    print(f'  [variant] tag={_variant_tag} — separate checkpoint directory')
if BG_QUAD_EPS > 0:
    print(f'  background quadratic eps={BG_QUAD_EPS}')

In [ ]:
# ── Cell 1: Environment ───────────────────────────────────────────
import os, sys, gc, shutil, subprocess, json, time, math
from pathlib import Path
from dataclasses import asdict

os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')

REPO_URL    = 'https://github.com/dimitarpg13/semsimula-paper.git'
REPO_BRANCH = 'main'

IN_COLAB = 'google.colab' in sys.modules
print(f'IN_COLAB = {IN_COLAB}')


def _sh(cmd):
    print(f'$ {cmd}')
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        raise RuntimeError(f'exit {r.returncode}: {cmd}')


if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    REPO_ROOT = Path('/content/semsimula-paper')
    if not (REPO_ROOT / '.git').exists():
        if REPO_ROOT.exists():
            shutil.rmtree(REPO_ROOT)
        _sh(f'git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} {REPO_ROOT}')
    else:
        try:
            _sh(f'git -C {REPO_ROOT} fetch --depth 1 origin {REPO_BRANCH}')
            _sh(f'git -C {REPO_ROOT} reset --hard origin/{REPO_BRANCH}')
        except RuntimeError as e:
            print(f'WARNING: repo refresh failed ({e}); using existing checkout.')

    _gdrive_name = 'semsimula_fock_multihead_openwebtext'
    if _variant_tag:
        _gdrive_name += f'_{_variant_tag}'
    GDRIVE_ROOT = Path(f'/content/drive/MyDrive/{_gdrive_name}')
    GDRIVE_ROOT.mkdir(parents=True, exist_ok=True)

    DATA_DIR = GDRIVE_ROOT / 'data'
    DATA_DIR.mkdir(exist_ok=True)
    repo_data = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'data'
    if repo_data.is_symlink():
        repo_data.unlink()
    elif repo_data.is_dir():
        shutil.rmtree(repo_data)
    repo_data.symlink_to(DATA_DIR)

    CKPT_DIR    = GDRIVE_ROOT / 'checkpoints'
    RESULTS_DIR = GDRIVE_ROOT / 'results'
    CKPT_DIR.mkdir(exist_ok=True)
    RESULTS_DIR.mkdir(exist_ok=True)

    _sh('pip install -q transformers huggingface_hub pyarrow')
else:
    REPO_ROOT = Path('.').resolve()
    while not (REPO_ROOT / '.git').exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent
    DATA_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'data'
    _local_phase = 'multihead' + (f'_{_variant_tag}' if _variant_tag else '')
    CKPT_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'scaleup' / 'results' / _local_phase / 'ckpts'
    RESULTS_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'scaleup' / 'results' / _local_phase
    for d in [DATA_DIR, CKPT_DIR, RESULTS_DIR]:
        d.mkdir(parents=True, exist_ok=True)

CA_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch'
for sub in ['', 'parf', 'multixi', 'scaleup', 'sarf_mass_variant', 'energetic_minima']:
    d = str(CA_DIR / sub) if sub else str(CA_DIR)
    if d not in sys.path:
        sys.path.insert(0, d)

CKPT_PREFIX   = 'fock_multihead_owt' + (f'_{_variant_tag}' if _variant_tag else '')
TOTAL_STEPS   = 200_000
CKPT_INTERVAL = 25_000
CKPT_STEPS    = list(range(CKPT_INTERVAL, TOTAL_STEPS + 1, CKPT_INTERVAL))

print(f'CKPT_DIR    = {CKPT_DIR}')
print(f'RESULTS_DIR = {RESULTS_DIR}')
print(f'Steps: {TOTAL_STEPS:,}  checkpoints at: {CKPT_STEPS}')

In [ ]:
# ── Cell 2: Checkpoint resolution + resume detection ─────────────
# Mirrors the Phase 4 resume logic: scan periodic step checkpoints in
# reverse order and compare step numbers against *_best.pt to pick the
# most advanced valid checkpoint for resumption.
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name}  ({props.total_memory/1e9:.1f} GB)')
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

resume_step = 0
resume_ckpt = None

# ── 1. Scan periodic step checkpoints ─────────────────────────────
for s in sorted(CKPT_STEPS, reverse=True):
    cand = CKPT_DIR / f'{CKPT_PREFIX}_step{s}.pt'
    if cand.exists():
        resume_ckpt = cand
        resume_step = s
        break

# ── 2. Also consider *_best.pt as a resume candidate ──────────────
# After a soft blowup, the periodic step checkpoint may contain diverged
# weights while *_best.pt still holds the pre-blowup best model.  We
# prefer *_best.pt whenever it is MORE RECENT than the latest step
# checkpoint, so a blowup between two step saves is always recoverable.
#
# Check two locations: the canonical {PREFIX}_best.pt AND any
# {PREFIX}_step{N}_best.pt files (which are created per-eval).
import glob as _glob, re as _re
_best_candidates = []
_canonical = CKPT_DIR / f'{CKPT_PREFIX}_best.pt'
if _canonical.exists():
    _best_candidates.append(_canonical)
for _f in sorted(CKPT_DIR.glob(f'{CKPT_PREFIX}_step*_best.pt')):
    _best_candidates.append(_f)

_best_path = None
_best_step_found = resume_step
for _cand in _best_candidates:
    try:
        _bd = torch.load(_cand, map_location='cpu', weights_only=False)
        _s = _bd.get('step', 0)
        _p = _bd.get('val_ppl', float('inf'))
        del _bd
        if _s > _best_step_found:
            _best_step_found = _s
            _best_path = _cand
            _best_ppl = _p
            print(f'  Found best candidate: {_cand.name} (step {_s:,}, PPL {_p:.2f})')
    except Exception as e:
        print(f'[warn] could not inspect {_cand.name}: {e}')

if _best_path is not None and _best_step_found > resume_step:
    print(f'Best checkpoint (step {_best_step_found:,}, PPL {_best_ppl:.2f}) is more recent '
          f'than latest periodic checkpoint (step {resume_step:,}) — resuming from best.')
    resume_ckpt = _best_path
    resume_step = _best_step_found

if resume_ckpt is not None:
    print(f'\nResuming from: {resume_ckpt.name}  (step {resume_step:,})')
    print(f'Remaining: {TOTAL_STEPS - resume_step:,} steps')
else:
    print('No checkpoint found — training from scratch.')
    print(f'Total: {TOTAL_STEPS:,} steps  Checkpoints every {CKPT_INTERVAL:,}')

In [ ]:
# ── Cell 3: Data loading (reuse cached OpenWebText from earlier phases) ──
from data_module import get_batch

MAX_TRAIN_TOKENS = 200_000_000
VAL_TOKENS       = 2_000_000
CHUNK_SIZE       = 50_000
VOCAB_SIZE       = 50257

from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained('gpt2')

train_cache = DATA_DIR / f'openwebtext_train_{MAX_TRAIN_TOKENS // 1_000_000}M.npy'
val_cache   = DATA_DIR / f'openwebtext_val_{VAL_TOKENS // 1_000_000}M.npy'

# Search prior-phase data caches to avoid re-downloading
for alt_name in [
    'semsimula_fock_structured_vtheta_owt_phase4',
    'semsimula_fock_gaussian_sarf_openwebtext_phase5',
    'semsimula_splm_openwebtext_phase4',
    'semsimula_splm_openwebtext_scaleup',
    'semsimula_parf_multixi_openwebtext_scaleup',
    'semsimula_fock_multixi_openwebtext_scaleup',
    'semsimula_splm_openwebtext',
]:
    if train_cache.exists():
        break
    if IN_COLAB:
        alt = Path(f'/content/drive/MyDrive/{alt_name}/data')
    else:
        alt = Path.home() / alt_name / 'data'
    alt_train = alt / f'openwebtext_train_{MAX_TRAIN_TOKENS // 1_000_000}M.npy'
    alt_val   = alt / f'openwebtext_val_{VAL_TOKENS // 1_000_000}M.npy'
    if alt_train.exists():
        import shutil
        print(f'Reusing data cache from {alt}')
        shutil.copy2(str(alt_train), str(train_cache))
        shutil.copy2(str(alt_val), str(val_cache))
        break

if train_cache.exists() and val_cache.exists():
    print('Loading cached OpenWebText tokens ...')
    train_ids = np.load(str(train_cache))
    val_ids   = np.load(str(val_cache))
    print(f'  train: {len(train_ids):,} tokens')
    print(f'  val:   {len(val_ids):,} tokens')
else:
    from datasets import load_dataset
    print(f'Streaming OpenWebText (target: {MAX_TRAIN_TOKENS:,} train + {VAL_TOKENS:,} val tokens) ...')
    ds = load_dataset('Skylion007/openwebtext', split='train', streaming=True, trust_remote_code=True)
    all_ids = []
    total = 0
    target = MAX_TRAIN_TOKENS + VAL_TOKENS
    chunk_texts = []
    n_docs = 0
    t0 = time.time()
    for example in ds:
        chunk_texts.append(example['text'])
        n_docs += 1
        if len(chunk_texts) >= CHUNK_SIZE:
            joined = '\n\n'.join(chunk_texts)
            chunk_ids = tok.encode(joined)
            all_ids.extend(chunk_ids)
            total = len(all_ids)
            elapsed = time.time() - t0
            print(f'  {n_docs:,} docs  {total:,} tokens  ({elapsed:.0f}s)', flush=True)
            chunk_texts = []
            del joined, chunk_ids
            if total >= target:
                break
    if chunk_texts:
        joined = '\n\n'.join(chunk_texts)
        all_ids.extend(tok.encode(joined))
        del joined, chunk_texts
    all_ids = np.array(all_ids, dtype=np.uint16)
    total = len(all_ids)
    print(f'Total streamed: {total:,} tokens from {n_docs:,} documents ({time.time() - t0:.0f}s)')
    val_ids   = all_ids[-VAL_TOKENS:]
    train_ids = all_ids[:-VAL_TOKENS]
    if len(train_ids) > MAX_TRAIN_TOKENS:
        train_ids = train_ids[:MAX_TRAIN_TOKENS]
    del all_ids
    np.save(str(train_cache), train_ids)
    np.save(str(val_cache), val_ids)
    print(f'  Cached: train={len(train_ids):,} -> {train_cache}')
    print(f'  Cached: val={len(val_ids):,}   -> {val_cache}')

print(f'train: {len(train_ids):,}   val: {len(val_ids):,}')

In [ ]:
# ── Cell 4b: PMI Spectral Diversity — K_MIX estimator ──────────────
# Computes the effective spectral rank of the PMI co-occurrence matrix
# to estimate how many Gaussian wells (K_MIX) the corpus needs.
# The ratio rank_eff(OWT) / rank_eff(TinyStories) gives a principled
# multiplier for scaling K_MIX from the TinyStories ablation baseline.

# TinyStories reference (compute once on Colab, then hard-code here):
RANK_EFF_TINYSTORIES = None  # Set after first TinyStories run, e.g. 85.3


def compute_pmi_effective_rank(token_ids, vocab_size, top_v=8192, window=5, n_components=512):
    """Compute effective spectral rank of the PMI matrix (Roy-Vetterli).

    rank_eff = exp(H(sigma_hat)) where sigma_hat is the normalised
    singular value spectrum and H is Shannon entropy.
    """
    print(f'  Building co-occurrence matrix (top_v={top_v}, window={window}) ...')
    token_counts = np.bincount(token_ids.astype(np.int64), minlength=vocab_size)
    top_v_ids = np.argsort(-token_counts)[:top_v]
    id_to_local = np.full(vocab_size, -1, dtype=np.int64)
    id_to_local[top_v_ids] = np.arange(top_v)

    cooc = np.zeros((top_v, top_v), dtype=np.float64)
    local_ids = id_to_local[token_ids.astype(np.int64)]
    for offset in range(1, window + 1):
        a, b = local_ids[:-offset], local_ids[offset:]
        valid = (a >= 0) & (b >= 0)
        np.add.at(cooc, (a[valid], b[valid]), 1.0)
    cooc = cooc + cooc.T

    row_sums = cooc.sum(axis=1, keepdims=True)
    total = cooc.sum()
    expected = row_sums * row_sums.T / total
    with np.errstate(divide='ignore', invalid='ignore'):
        pmi = np.log(cooc / np.maximum(expected, 1e-12))
    pmi = np.nan_to_num(pmi, nan=0.0, posinf=0.0, neginf=-20.0)
    np.fill_diagonal(pmi, 0.0)

    print(f'  Computing truncated SVD (k={n_components}) ...')
    from scipy.sparse.linalg import svds
    from scipy.sparse import csr_matrix
    pmi_sparse = csr_matrix(pmi)
    _, s, _ = svds(pmi_sparse, k=min(n_components, top_v - 1))
    s = np.sort(s)[::-1]

    s_norm = s / s.sum()
    s_norm = s_norm[s_norm > 1e-12]
    entropy = -np.sum(s_norm * np.log(s_norm))
    rank_eff = np.exp(entropy)
    return rank_eff, s


print('Computing PMI spectral diversity for OpenWebText ...')
t_pmi = time.time()
rank_eff_owt, sv_owt = compute_pmi_effective_rank(train_ids, VOCAB_SIZE)
print(f'  Done in {time.time() - t_pmi:.1f}s')
print(f'\n  rank_eff(OpenWebText) = {rank_eff_owt:.1f}')
print(f'  Top-5 singular values: {sv_owt[:5]}')
print(f'  SV decay ratio (s[0]/s[50]): {sv_owt[0]/sv_owt[min(50, len(sv_owt)-1)]:.1f}')

if RANK_EFF_TINYSTORIES is not None:
    diversity_ratio = rank_eff_owt / RANK_EFF_TINYSTORIES
    suggested_k = max(8, round(8 * diversity_ratio))
    print(f'\n  rank_eff(TinyStories) = {RANK_EFF_TINYSTORIES:.1f}  (reference)')
    print(f'  Diversity ratio: {diversity_ratio:.2f}x')
    print(f'  Suggested K_MIX = round(8 * {diversity_ratio:.2f}) = {suggested_k}')
    print(f'\n  >> To use: set K_MIX = {suggested_k} in Cell 0 (Configuration)')
else:
    print(f'\n  [info] RANK_EFF_TINYSTORIES not set.')
    print(f'  Run this same function on TinyStories train_ids, then set')
    print(f'  RANK_EFF_TINYSTORIES = <value> at the top of this cell.')
    print(f'  Then re-run to get the diversity ratio and suggested K_MIX.')

In [ ]:
# ── Cell 4: Model config + multi-head V_phi + Gaussian/SARF V_theta ──
import math
from model_fock_parf_multixi import FockMultiXiPARFLM, FockMultiXiPARFConfig
import model_fock_parf_v2
import model_parf_multixi
import model_parf
import model_parf_sparse

if XI_OVERRIDE == 5:
    XI_CHANNELS    = 5
    XI_ALPHA_INITS = [0.25, 0.50, 0.75, 0.95, 0.99]
    print(f'Xi override: {XI_CHANNELS} channels, alphas={XI_ALPHA_INITS}')
    print(f'  Horizons: ~{[round(1/(1-a),1) for a in XI_ALPHA_INITS]} tokens')
elif XI_OVERRIDE is not None:
    raise ValueError(f'Unsupported XI_OVERRIDE={XI_OVERRIDE}; use None or 5')
else:
    XI_CHANNELS    = 4
    XI_ALPHA_INITS = [0.25, 0.50, 0.75, 0.95]
LAMBDA_V       = 1e-2
BLOCK_SIZE     = 512

LOGFREQ_PATH = CA_DIR / 'scaleup' / 'results' / 'logfreq_surprisal_openwebtext.npy'
DRIVE_LOGFREQ = RESULTS_DIR / 'logfreq_surprisal_openwebtext.npy'

if LOGFREQ_PATH.exists():
    LOGFREQ_FILE = LOGFREQ_PATH
elif DRIVE_LOGFREQ.exists():
    LOGFREQ_FILE = DRIVE_LOGFREQ
else:
    counts = np.bincount(train_ids.astype(np.int64), minlength=VOCAB_SIZE).astype(np.float64)
    p = (counts + 1.0) / (counts.sum() + VOCAB_SIZE)
    surprisal = (-np.log(p)).astype(np.float32)
    LOGFREQ_FILE = DRIVE_LOGFREQ
    LOGFREQ_FILE.parent.mkdir(parents=True, exist_ok=True)
    np.save(LOGFREQ_FILE, surprisal)

print(f'Logfreq: {LOGFREQ_FILE}')

# Adaptive architecture tiers
ARCH_TIERS = [
    (384, 16, 32),
    (384, 12, 16),
    (256, 16, 16),
    (256,  8, 16),
]


def make_config(d, L, n_registers):
    return FockMultiXiPARFConfig(
        vocab_size=VOCAB_SIZE, d=d, max_len=1024,
        L=L, v_hidden=1024, v_depth=3, dt=1.0,
        mass_mode='logfreq',
        logfreq_path=str(LOGFREQ_FILE),
        logfreq_init_alpha=0.1,
        init_gamma=1.0,
        fixed_gamma=0.30,
        causal_force=True,
        ln_after_step=True,
        xi_channels=XI_CHANNELS,
        xi_alpha_inits=XI_ALPHA_INITS,
        xi_learnable=True,
        xi_alpha_init_mode='explicit',
        v_phi_kind='structural_competitive',
        v_phi_phi_hidden=128,
        v_phi_theta_hidden=128,
        top_k=TOP_K,
        v_phi_n_heads=V_PHI_N_HEADS,
        use_output_bias=USE_OUTPUT_BIAS,
        tie_embeddings=TIE_EMBEDDINGS,
        score_head_hidden=32,
        gumbel_tau_init=1.0,
        gumbel_tau_min=0.3,
        gumbel_noise=True,
        use_gathered_v_phi=True,
        use_layer_checkpoint=True,
        ln_before_distance=True,
        per_layer_v_phi_scale=True,
        fock_version='v2',
        n_registers=n_registers,
        register_salience_decay=0.5,
        register_salience_threshold=0.005,
        creation_gate_hidden=64,
        stack_discipline=True,
        d_k=64,
        tau_create_init=8.0,
        reverse_channel=True,
        per_register_tau=True,
        per_register_keys=True,
        ortho_register_init=True,
    )


def build_structured_vtheta(model, d, variant, device):
    """Swap model.V_theta with a structured variant.

    Supported variants:
      - 'gaussian': MixtureGaussianVTheta (bounded wells)
        Fixes: init_log_precision=-log(d), precision_max=2/d
      - 'sarf': SARFGaussianVTheta (bounded, frozen PMI anchors)
        Fixes: init_log_sigma=ln(sqrt(d)), log_sigma_max=0.5*log(d)+1.0
      - 'sq3': MixtureQuadraticVTheta (unbounded, with curvature clamp)
        Fixes: curvature_max, log1p(V^2) penalty (applied in loss), force_clamp_max
    """
    xi_d = XI_CHANNELS * d
    from model_gaussian_vtheta import (
        MixtureGaussianVTheta, SARFGaussianVTheta, GaussianVThetaMultiXiAdapter,
    )

    if variant == 'gaussian':
        # init_log_precision = -log(d): ensures sigma_eff ≈ sqrt(d) from step 1,
        # matching the LN-normalised hidden-state scale (||h_L|| ≈ sqrt(d)).
        _init_log_prec = -math.log(d)
        # precision_max = 2/d: prevents a_k → ∞ (delta spikes, force divergence).
        _prec_max = 2.0 / d

        inner = MixtureGaussianVTheta(
            d=d, K=K_MIX, w_scale=W_SCALE, xi_d=xi_d,
            init_log_precision=_init_log_prec,
            precision_max=_prec_max,
        )
        model.V_theta = GaussianVThetaMultiXiAdapter(inner, K=XI_CHANNELS, d=d).to(device)

        _sigma_eff_init = 1.0 / (math.exp(_init_log_prec) ** 0.5)
        _sigma_min = 1.0 / (_prec_max ** 0.5)
        print(f'V_theta -> Gaussian(K={K_MIX}, w_scale={W_SCALE},'
              f' sigma_eff_init={_sigma_eff_init:.2f}, sigma_min={_sigma_min:.2f})')

    elif variant == 'sarf':
        # Compute SARF anchors from OWT PMI
        WINDOW = 5
        TOP_V = 8192
        token_counts = np.bincount(train_ids.astype(np.int64), minlength=VOCAB_SIZE)
        top_v_ids = np.argsort(-token_counts)[:TOP_V]
        id_to_local = np.full(VOCAB_SIZE, -1, dtype=np.int64)
        id_to_local[top_v_ids] = np.arange(TOP_V)

        cooc = np.zeros((TOP_V, TOP_V), dtype=np.float64)
        local_ids = id_to_local[train_ids.astype(np.int64)]
        for offset in range(1, WINDOW + 1):
            a = local_ids[:-offset]
            b = local_ids[offset:]
            valid = (a >= 0) & (b >= 0)
            np.add.at(cooc, (a[valid], b[valid]), 1.0)
        cooc = cooc + cooc.T

        row_sums = cooc.sum(axis=1, keepdims=True)
        total = cooc.sum()
        expected = row_sums * row_sums.T / total
        with np.errstate(divide='ignore', invalid='ignore'):
            pmi = np.log(cooc / np.maximum(expected, 1e-12))
        pmi = np.nan_to_num(pmi, nan=0.0, posinf=0.0, neginf=-20.0)

        np.fill_diagonal(pmi, -np.inf)
        pmi_peaks = pmi.max(axis=1)
        anchor_local_ids = np.argsort(-pmi_peaks)[:SARF_N_ANCHORS]
        anchor_token_ids = top_v_ids[anchor_local_ids]
        anchor_positions = model.E.weight.data[anchor_token_ids].detach().clone()
        print(f'SARF anchors: {SARF_N_ANCHORS} PMI-peak tokens selected')
        print(f'  PMI peak range: [{pmi_peaks[anchor_local_ids[-1]]:.2f}, '
              f'{pmi_peaks[anchor_local_ids[0]]:.2f}]')

        # log_sigma_max = 0.5*log(d)+1.0: caps sigma at ≈e×sqrt(d), preventing
        # well deactivation (sigma drift → flat wells → v_reg collapse).
        _log_sigma_max = 0.5 * math.log(d) + 1.0
        # init_log_sigma = 2.77 → sigma ≈ 16 ≈ sqrt(d), matching LN-scale h_L.
        _init_log_sigma = math.log(d) / 2.0  # ln(sqrt(d)) ≈ 2.77 for d=256

        inner = SARFGaussianVTheta(
            d=d, anchor_positions=anchor_positions, xi_d=xi_d, w_scale=W_SCALE,
            init_log_sigma=_init_log_sigma,
            log_sigma_max=_log_sigma_max,
        )
        model.V_theta = GaussianVThetaMultiXiAdapter(inner, K=XI_CHANNELS, d=d).to(device)

        # Re-normalise anchors to match ln_after_step hidden-state scale.
        # ln_after_step=True normalises h to unit variance per dim after every
        # Verlet step → ||h_L|| ≈ sqrt(d).  Raw embedding anchors have norm ≈ 0.3.
        # Without normalisation, ||h_L - a_j|| ≈ sqrt(d) >> 0.3 → Gaussian bumps vanish.
        with torch.no_grad():
            a = model.V_theta.inner.anchors
            a = (a - a.mean(dim=-1, keepdim=True)) / (a.std(dim=-1, keepdim=True) + 1e-5)
            model.V_theta.inner.anchors.copy_(a)

        print(f'V_theta -> SARF Gaussian(N_S={SARF_N_ANCHORS})')
        print(f'  Anchors re-normalised: norm={a.norm(dim=-1).mean():.2f}'
              f'  (target ~{d**0.5:.1f})')
        print(f'  sigma_init={model.V_theta.inner.sigma.mean():.2f}'
              f'  sigma_max={math.exp(_log_sigma_max):.2f}')

    elif variant == 'sq3':
        from model_structured_vtheta import MixtureQuadraticVTheta
        from model_structured_vtheta_multixi import StructuredVThetaMultiXiAdapter

        inner = MixtureQuadraticVTheta(
            d=d, K=K_MIX, tau=SQ3_TAU, init_a_bias=0.0,
            xi_d=xi_d,
        )

        if SQ3_CURV_MAX is not None:
            _orig_components = inner._components
            def _clamped_components(xi, _fn=_orig_components, _cmax=SQ3_CURV_MAX):
                mu, a, log_pi = _fn(xi)
                return mu, a.clamp(max=_cmax), log_pi
            inner._components = _clamped_components

        model.V_theta = StructuredVThetaMultiXiAdapter(
            inner, K=XI_CHANNELS, d=d,
        ).to(device)

        print(f'V_theta -> SQ3 Mixture(K={K_MIX}, tau={SQ3_TAU})')
        print(f'  xi_d={xi_d}, curvature_max={SQ3_CURV_MAX}')
        n_sq3 = sum(p.numel() for p in model.V_theta.parameters())
        print(f'  SQ3 params: {n_sq3:,}')

    else:
        raise ValueError(f'Unknown V_theta variant: {variant}')


# ── Try architecture tiers ─────────────────────────────────────────
model = None
model_cfg = None
for d, L, M in ARCH_TIERS:
    try:
        cfg = make_config(d, L, M)
        mdl = FockMultiXiPARFLM(cfg).to(DEVICE)
        n_v_theta_mlp = sum(p.numel() for p in mdl.V_theta.parameters())
        build_structured_vtheta(mdl, d, V_THETA_VARIANT, DEVICE)
        n = mdl.num_params()
        n_v_theta = sum(p.numel() for p in mdl.V_theta.parameters())
        _vt_label = V_THETA_VARIANT.upper() if V_THETA_VARIANT == 'sq3' else 'Gaussian'
        print(f'Trying d={d} L={L} M={M} -> {n:,} params '
              f'(V_theta {n_v_theta_mlp:,} MLP -> {n_v_theta:,} {_vt_label})')
        if DEVICE == 'cuda':
            _rng = np.random.default_rng(42)
            _xb, _yb = get_batch(train_ids, 2, BLOCK_SIZE, _rng)
            _x = torch.from_numpy(_xb).to(DEVICE)
            _y = torch.from_numpy(_yb).to(DEVICE)
            _, _loss = mdl(_x, _y)
            _loss.backward()
            mdl.zero_grad(set_to_none=True)
            del _x, _y, _xb, _yb, _loss
            torch.cuda.empty_cache()
            print(f'OOM probe passed (batch=2)')
        model = mdl
        model_cfg = cfg
        break
    except RuntimeError as e:
        if 'out of memory' in str(e).lower():
            print(f'  OOM at d={d} L={L} M={M} — trying next tier ...')
            del mdl
            gc.collect()
            if DEVICE == 'cuda':
                torch.cuda.empty_cache()
            continue
        raise

if model is None:
    raise RuntimeError('All architecture tiers OOMed.')

# ── Initialise output bias to log unigram frequency (D0.4 long-tail fix) ──
# Hands the frequency prior to b_v so the conservative dynamics stop spending
# hidden capacity on it.  No-op when USE_OUTPUT_BIAS is False.  On resume the
# loaded checkpoint's bias overwrites this (strict=False keeps it for fresh
# runs / when an old no-bias checkpoint is loaded into a bias-enabled model).
if USE_OUTPUT_BIAS:
    _ob_counts = np.bincount(train_ids.astype(np.int64), minlength=VOCAB_SIZE)
    model.init_output_bias_from_logfreq(_ob_counts)
    print(f'Output bias <- log-unigram-freq  '
          f'(b range [{model.out_bias.min().item():.2f}, '
          f'{model.out_bias.max().item():.2f}])')

# ── Auto batch size ────────────────────────────────────────────────
BATCH_SIZE = 4
GRAD_ACCUM = 2
if DEVICE == 'cuda':
    for bs in [8, 6, 4]:
        try:
            _rng = np.random.default_rng(42)
            _xb, _yb = get_batch(train_ids, bs, BLOCK_SIZE, _rng)
            _x = torch.from_numpy(_xb).to(DEVICE)
            _y = torch.from_numpy(_yb).to(DEVICE)
            _, _loss = model(_x, _y)
            _loss.backward()
            model.zero_grad(set_to_none=True)
            del _x, _y, _xb, _yb, _loss
            torch.cuda.empty_cache()
            BATCH_SIZE = bs
            GRAD_ACCUM = max(1, 8 // bs)
            print(f'Auto batch: {bs} x accum={GRAD_ACCUM} (eff={bs*GRAD_ACCUM})')
            break
        except RuntimeError:
            if DEVICE == 'cuda':
                torch.cuda.empty_cache()
            continue

EFFECTIVE_BATCH = BATCH_SIZE * GRAD_ACCUM
n_params = model.num_params()
n_v_theta = sum(p.numel() for p in model.V_theta.parameters())
IS_STRUCTURED = V_THETA_VARIANT in ('gaussian', 'sarf', 'sq3')

_vtheta_name = {'gaussian': 'Gaussian', 'sarf': 'SARF', 'sq3': 'SQ3'}[V_THETA_VARIANT]
print(f'\nModel: FockMultiXiPARFLM v2.1 + {_vtheta_name} V_theta (Phase 5)')
print(f'  params: {n_params:,}  (V_theta: {n_v_theta:,})')
print(f'  d={model_cfg.d}  L={model_cfg.L}  M={model_cfg.n_registers}')
print(f'  V_theta={V_THETA_VARIANT}  lambda_V={LAMBDA_V}')
print(f'  V_phi=structural_competitive x {V_PHI_N_HEADS} head(s)  top_k={TOP_K}')
print(f'  batch={BATCH_SIZE} x accum={GRAD_ACCUM} (eff={EFFECTIVE_BATCH})')
print(f'  IS_STRUCTURED={IS_STRUCTURED}')

In [ ]:
# ── Cell 5: Training loop ─────────────────────────────────────────
# Xi diagnostics disabled: run_xi_diagnostics calls model.eval() and on failure
# never restores model.train(), leaving the model locked in eval mode for all
# subsequent steps (v_reg collapses to 0).  Re-enable only after the fix in
# xi_bottleneck_diagnostics.py wraps the call in try/finally + model.train().
# from xi_bottleneck_diagnostics import run_xi_diagnostics

LR            = 1.2e-4    # matched to Phase 4; 2e-4 caused doom loop at d=384
WEIGHT_DECAY  = 0.01
WARMUP_STEPS  = 4000      # shorter warmup since peak LR is lower
GRAD_CLIP     = 1.0       # Phase 4 value; 0.5 was too aggressive with LR=2e-4
EVAL_INTERVAL = 2000
EVAL_ITERS    = 40
LOG_INTERVAL  = 200
# DIAG_INTERVAL = 10000   # disabled (see comment above)
SEED          = 0

# SQ3 override: unbounded potential requires tighter LR + lower watchdog
# patience to catch genuine divergence early.  Based on Section 13–14
# analysis of scale-diversity instability for non-bounded V_theta.
if V_THETA_VARIANT == 'sq3':
    LR        = 8e-5      # 0.67x Gaussian LR — matches SQ3 stability bound
    GRAD_CLIP = 0.5       # SQ3 forces grow ∝ a_k, tighter clip prevents runaway
    print(f'[SQ3 override] LR={LR}, GRAD_CLIP={GRAD_CLIP}')

# Watchdog: raised from 20→40 at step ~40K because the model entered a
# plateau-with-instability cycle: transient gradient spikes (recoverable
# via grad_clip=1.0) kept triggering the watchdog and reloading, wasting
# steps without improving PPL.  Threshold 40 catches genuine blowups
# (sustained EMA > 40) while letting the model work through rough patches.
GRAD_NORM_EMA_ALPHA = 0.05
GRAD_NORM_EMA_THRESHOLD = 40.0
GRAD_NORM_EMA_PATIENCE = 200

torch.manual_seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)


def lr_schedule(step):
    if step < WARMUP_STEPS:
        return LR * (step + 1) / WARMUP_STEPS
    progress = (step - WARMUP_STEPS) / max(TOTAL_STEPS - WARMUP_STEPS, 1)
    return LR * 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))


def forward_with_vreg(x, targets, lambda_v):
    h0 = model._embed(x)
    h_L, _ = model._stack_forward(h0, x, return_trajectory=False)
    # Use the model read-out head (respects USE_OUTPUT_BIAS / TIE_EMBEDDINGS);
    # a bare h_L @ E^T here would silently bypass the output bias / untied W_out.
    logits = model.compute_logits(h_L)
    loss_ntp = F.cross_entropy(
        logits.reshape(-1, model_cfg.vocab_size),
        targets.reshape(-1),
    )
    v_reg_value = torch.tensor(0.0, device=x.device)
    if lambda_v > 0:
        xis = model.xi_module(h_L.detach())
        V_vals = model.V_theta(xis, h_L)
        if V_THETA_VARIANT == 'sq3':
            # SQ3 is unbounded: log1p(V^2) prevents gradient explosion from large V
            v_reg_value = torch.log1p(V_vals ** 2).mean()
        else:
            v_reg_value = (V_vals ** 2).mean()
        if BG_QUAD_EPS > 0:
            bg = BG_QUAD_EPS * (h_L ** 2).sum(dim=-1, keepdim=True).mean()
            loss = loss_ntp + lambda_v * v_reg_value + bg
        else:
            loss = loss_ntp + lambda_v * v_reg_value
    else:
        loss = loss_ntp
    return loss, loss_ntp, v_reg_value


@torch.no_grad()
def evaluate():
    model.eval()
    losses = []
    for _ in range(EVAL_ITERS):
        xb, yb = get_batch(val_ids, BATCH_SIZE, BLOCK_SIZE, rng)
        x = torch.from_numpy(xb).to(DEVICE)
        y = torch.from_numpy(yb).to(DEVICE)
        with torch.enable_grad():
            _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses))


def save_checkpoint(step_num, val_loss_val, tag_suffix=''):
    ckpt = {
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optim.state_dict(),
        'model_cfg': asdict(model_cfg),
        'train_cfg': {
            'batch_size': BATCH_SIZE, 'block_size': BLOCK_SIZE,
            'grad_accum': GRAD_ACCUM, 'effective_batch': EFFECTIVE_BATCH,
            'steps': TOTAL_STEPS, 'lr': LR, 'weight_decay': WEIGHT_DECAY,
            'warmup_steps': WARMUP_STEPS, 'grad_clip': GRAD_CLIP,
            'lambda_v': LAMBDA_V, 'v_theta_variant': V_THETA_VARIANT,
        },
        'step': step_num,
        'val_loss': val_loss_val,
        'val_ppl': math.exp(val_loss_val),
        'gamma': model.gamma.item(),
        'xi_alphas': model.xi_alpha_values(),
        'variant': f'fock_parf_multixi_v2.1_{V_THETA_VARIANT}',
        'corpus': 'openwebtext',
        'phase': 5,
        'seed': SEED,
    }
    fname = f'{CKPT_PREFIX}_step{step_num}{tag_suffix}.pt'
    path = CKPT_DIR / fname
    torch.save(ckpt, path)
    print(f'  Checkpoint saved: {path}  (PPL={math.exp(val_loss_val):.2f})')
    if '_best' in tag_suffix:
        canonical = CKPT_DIR / f'{CKPT_PREFIX}_best.pt'
        import shutil
        shutil.copy2(path, canonical)
        print(f'  Canonical best: {canonical}')
    return path


# ── Optimizer ──
optim = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=LR,
    weight_decay=WEIGHT_DECAY, betas=(0.9, 0.95),
)

# ── Resume ──
if resume_ckpt is not None and resume_step < TOTAL_STEPS:
    print(f'Resuming from checkpoint at step {resume_step:,}: {resume_ckpt}')
    ckpt_data = torch.load(resume_ckpt, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt_data['model_state_dict'], strict=False)
    if 'optimizer_state_dict' in ckpt_data:
        try:
            optim.load_state_dict(ckpt_data['optimizer_state_dict'])
            print('  Optimizer state restored.')
        except (ValueError, KeyError) as e:
            print(f'  [info] Optimizer state incompatible, starting fresh: {e}')
    prev_ppl = ckpt_data.get('val_ppl', float('nan'))
    print(f'  Model loaded. Previous PPL: {prev_ppl:.2f}')
    del ckpt_data
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

# ── Training state ──
log_path = RESULTS_DIR / 'training_log.jsonl'
log_f = log_path.open('a')

t0 = time.time()
model.train()
run_ntp = 0.0
run_vreg = 0.0
n_run = 0
n_skipped = 0

best_val_ppl = float('inf')
_best_ckpt_path = CKPT_DIR / f'{CKPT_PREFIX}_best.pt'

# If canonical _best.pt doesn't exist, find the latest _step*_best.pt
if not _best_ckpt_path.exists():
    _step_bests = sorted(CKPT_DIR.glob(f'{CKPT_PREFIX}_step*_best.pt'))
    if _step_bests:
        _best_ckpt_path = _step_bests[-1]
        print(f'No canonical _best.pt; using {_best_ckpt_path.name}')
        import shutil
        _canonical = CKPT_DIR / f'{CKPT_PREFIX}_best.pt'
        shutil.copy2(_best_ckpt_path, _canonical)
        _best_ckpt_path = _canonical
        print(f'  Copied to canonical: {_canonical.name}')

if _best_ckpt_path.exists():
    try:
        _bd = torch.load(_best_ckpt_path, map_location='cpu', weights_only=False)
        best_val_ppl = _bd.get('val_ppl', float('inf'))
        print(f'Restored running best PPL: {best_val_ppl:.2f}')
        del _bd
    except Exception as e:
        print(f'[warn] {e}')

# ── EMA watchdog (relaxed for Gaussian V) ──
_grad_norm_ema = 0.0
_grad_norm_above_thresh = 0

def _reload_best():
    if not _best_ckpt_path.exists():
        return resume_step
    ckpt = torch.load(_best_ckpt_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'], strict=False)
    try:
        optim.load_state_dict(ckpt['optimizer_state_dict'])
    except (ValueError, KeyError):
        pass
    s = ckpt.get('step', 0)
    p = ckpt.get('val_ppl', float('nan'))
    del ckpt
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    print(f'[watchdog] Reloaded best: step {s:,} PPL {p:.2f}')
    return s

steps_this_session = 0

print(f'\n{"="*60}')
print(f'Phase 5 ({_vtheta_name} {V_THETA_VARIANT}): steps {resume_step+1:,} -> {TOTAL_STEPS:,}')
print(f'  batch={BATCH_SIZE} x accum={GRAD_ACCUM} (eff={EFFECTIVE_BATCH})')
print(f'  block={BLOCK_SIZE}  lr={LR}  warmup={WARMUP_STEPS}  grad_clip={GRAD_CLIP}')
print(f'  d={model_cfg.d}  L={model_cfg.L}  M={model_cfg.n_registers}')
print(f'  V_theta={V_THETA_VARIANT}  params={n_params:,}')
print(f'  watchdog: threshold={GRAD_NORM_EMA_THRESHOLD} patience={GRAD_NORM_EMA_PATIENCE}')
print(f'{"="*60}\n')

for step in range(resume_step, TOTAL_STEPS):
    lr_now = lr_schedule(step)
    for g in optim.param_groups:
        g['lr'] = lr_now

    optim.zero_grad(set_to_none=True)
    accum_ntp = 0.0
    accum_vreg = 0.0
    for _acc in range(GRAD_ACCUM):
        xb, yb = get_batch(train_ids, BATCH_SIZE, BLOCK_SIZE, rng)
        x = torch.from_numpy(xb).to(DEVICE)
        y = torch.from_numpy(yb).to(DEVICE)
        loss, loss_ntp, v_reg = forward_with_vreg(x, y, LAMBDA_V)
        (loss / GRAD_ACCUM).backward()
        accum_ntp  += loss_ntp.item()       / GRAD_ACCUM
        accum_vreg += float(v_reg.detach()) / GRAD_ACCUM

    grad_norm = nn.utils.clip_grad_norm_(
        [p for p in model.parameters() if p.requires_grad], GRAD_CLIP,
    )
    if torch.isfinite(grad_norm) and math.isfinite(accum_ntp):
        optim.step()
        # Projected gradient: clamp log_sigma back into the feasible set so Adam's
        # momentum cannot push sigma past log_sigma_max even when the forward-pass
        # clamp makes the gradient surface flat beyond the boundary.
        if IS_STRUCTURED and hasattr(getattr(model.V_theta, 'inner', None), 'clamp_params'):
            model.V_theta.inner.clamp_params()
    else:
        n_skipped += 1
        optim.zero_grad(set_to_none=True)

    # ── Watchdog ──
    _raw_gn = float(grad_norm)
    _grad_norm_ema = (1 - GRAD_NORM_EMA_ALPHA) * _grad_norm_ema + GRAD_NORM_EMA_ALPHA * _raw_gn
    if _grad_norm_ema > GRAD_NORM_EMA_THRESHOLD:
        _grad_norm_above_thresh += 1
    else:
        _grad_norm_above_thresh = 0

    if _grad_norm_above_thresh >= GRAD_NORM_EMA_PATIENCE:
        print(f'\n[watchdog] EMA grad_norm={_grad_norm_ema:.1f} > {GRAD_NORM_EMA_THRESHOLD} '
              f'for {_grad_norm_above_thresh} steps at step {step+1}.')
        _reload_best()
        _grad_norm_ema = 0.0
        _grad_norm_above_thresh = 0
        n_skipped += 1

    run_ntp += accum_ntp
    run_vreg += accum_vreg
    n_run += 1
    steps_this_session += 1

    if (step + 1) % LOG_INTERVAL == 0:
        avg_ntp = run_ntp / n_run
        avg_vreg = run_vreg / n_run
        run_ntp, run_vreg, n_run = 0.0, 0.0, 0
        elapsed = time.time() - t0
        sec_per_step = elapsed / steps_this_session
        remaining = (TOTAL_STEPS - step - 1) * sec_per_step
        alphas = model.xi_alpha_values()
        alpha_str = ','.join(f'{a:.3f}' for a in alphas)
        print(
            f'step {step+1:7d}/{TOTAL_STEPS}  '
            f'ntp={avg_ntp:.4f}  v_reg={avg_vreg:.4f}  lr={lr_now:.2e}  '
            f'grad={float(grad_norm):.2f}  gamma={model.gamma.item():.3f}  '
            f'alpha=[{alpha_str}]  '
            f'{elapsed:.0f}s  (~{remaining/3600:.1f}h remaining)'
        )
        log_f.write(json.dumps({
            'step': step + 1, 'train_loss': avg_ntp, 'v_reg': avg_vreg,
            'lr': lr_now, 'grad_norm': float(grad_norm),
            'gamma': model.gamma.item(), 'xi_alphas': alphas,
            'elapsed_sec': elapsed, 'sec_per_step': sec_per_step,
        }) + '\n')
        log_f.flush()

    if (step + 1) % EVAL_INTERVAL == 0:
        val_loss = evaluate()
        val_ppl = math.exp(val_loss)
        is_best = val_ppl < best_val_ppl
        if is_best:
            best_val_ppl = val_ppl
        elapsed = time.time() - t0
        marker = '*** NEW BEST ***' if is_best else ''
        print(f'>>> EVAL step {step+1:,}  val_loss={val_loss:.4f}  '
              f'val_ppl={val_ppl:.2f}  best={best_val_ppl:.2f}  '
              f'{marker}  ({elapsed:.0f}s)')
        log_f.write(json.dumps({
            'step': step + 1, 'val_loss': val_loss,
            'val_ppl': val_ppl, 'best_ppl': best_val_ppl,
        }) + '\n')
        log_f.flush()
        if is_best:
            save_checkpoint(step + 1, val_loss, tag_suffix='_best')
        # Xi diagnostics block disabled — model.train() was not called on failure,
        # leaving the model in eval() mode and collapsing v_reg to 0.
        # if IS_STRUCTURED and (step + 1) % DIAG_INTERVAL == 0:
        #     try:
        #         _diag_xb, _diag_yb = get_batch(val_ids, BATCH_SIZE, BLOCK_SIZE, rng)
        #         _diag_x = torch.from_numpy(_diag_xb).to(DEVICE)
        #         _diag = run_xi_diagnostics(
        #             model, _diag_x, model.E.weight.data,
        #             K_xi=XI_CHANNELS, d=model_cfg.d, tokenizer=tok,
        #         )
        #         print(_diag['summary'])
        #         log_f.write(json.dumps({
        #             'step': step + 1,
        #             'diag_well_collapse': _diag['well_collapse'],
        #             'diag_xi_sensitivity': _diag['xi_sensitivity'],
        #             'diag_weight_entropy': _diag['weight_entropy'],
        #             'diag_xi_alphas': _diag['xi_alphas'],
        #             'diag_xi_horizons': _diag['xi_horizons'],
        #         }) + '\n')
        #         log_f.flush()
        #         del _diag_x, _diag_xb, _diag_yb, _diag
        #         model.train()
        #     except Exception as e:
        #         print(f'[diag] Xi diagnostics failed: {e}')

    if (step + 1) in set(CKPT_STEPS):
        if (step + 1) % EVAL_INTERVAL != 0:
            val_loss = evaluate()
            val_ppl = math.exp(val_loss)
        save_checkpoint(step + 1, val_loss)

log_f.close()
print(f'\nPhase 5 training complete. Best PPL: {best_val_ppl:.2f}')